In [2]:
import random
random.seed(42)

import numpy as np
import pyomo.environ as pyo
import pandas as pd

from pyomo.environ import *

np.random.seed(43)

In [3]:
n_sc = 300

In [4]:
model = pyo.ConcreteModel(name="BasicProblem")

model.W = RangeSet(1, n_sc)  # scenario indices
model.N = pyo.RangeSet(0,3) # n weeks prediction
#model.var_ind = pyo.Set(initialize=contextual_sample.columns.sort_values())

# --- PARAMETERS ---
model.price_min = Param(initialize=5.0, mutable=True)
model.price_max = Param(initialize=20.0, mutable=True)
model.cost_per_unit = Param(initialize=3.0, mutable=True)
model.pi = Param(model.W, initialize={w: 1/n_sc for w in range(1, n_sc+1)}, mutable=True)

# --- DECISION VARIABLES ---
model.price = Var(model.N, within=Reals)
#model.demand = Var(model.N, within=NonNegativeReals)
#model.demand = pyo.Var(pyo.Any, dense=False, domain=pyo.Reals)
model.demand = pyo.Var(model.W, model.N, domain=NonNegativeReals)

model.profit = Var(model.W, within=Reals)

# --- CONSTRAINTS ---

# Link profit linearly
def profit_constraint_rule(model, w):
    return model.profit[w] == sum((model.price[n] - model.cost_per_unit) * model.demand[w, n] for n in model.N)
model.profit_constraint = Constraint(model.W, rule=profit_constraint_rule)

def demand_limit_rule(model, n):
    return sum(model.demand[w, n] for w in model.W) <= 100 - 3 * model.price[n]
model.demand_limit = Constraint(model.N, rule=demand_limit_rule)


def price_lower_rule(model, n):
    return model.price[n] >= model.price_min
model.price_lower = Constraint(model.N, rule=price_lower_rule)

def price_upper_rule(model, n):
    return model.price[n] <= model.price_max
model.price_upper = Constraint(model.N, rule=price_upper_rule)

# Objective: maximize expected profit
def objective(model):
    return sum(model.pi[w] * model.profit[w] for w in model.W)
model.OBJ = Objective(rule=objective, sense=maximize)


In [5]:
cons = pd.read_csv('constraints_1_50.csv', index_col=0)

In [6]:
# --- SOLVE ---
solver = SolverFactory('glpk')  # or 'cbc', 'ipopt', etc.
result = solver.solve(model)

# --- OUTPUT ---
print("Status:", result.solver.status)
print("Termination Condition:", result.solver.termination_condition)
print("Optimal price:", value(model.price))
print("Predicted demand:", value(model.demand))
print("Maximum profit:", value(model.profit))

ValueError: Model constraint (profit_constraint[1]) contains nonlinear terms that cannot be written to LP format